# Privacy Lens · 실사 평가 노트북

팀원이 찍은 실사 사진으로 **지금 서비스가 무엇을 놓치는지, 왜 놓치는지**를 잽니다. 사진은 드라이브에만 두고 저장소에는 올리지 않습니다.
자세한 절차는 `docs/REAL_PHOTO_GUIDE.md`.

| 하고 싶은 일 | 실행할 칸 |
|---|---|
| 처음 준비(폴더·정답 입력 도구·소품) | 0 → 1 → 2 |
| 평가하고 기록 남기기 | 0 → 1 → 3 (→ 4: YOLO 켠 것과 비교) |
| 기록만 보기 | 0 → 1 → 5 |

GPU 는 필요 없습니다(CPU 로 사진 30장에 몇 분). YOLO 비교(4번)만 GPU 가 있으면 빠릅니다.

## 0. 설정

In [ ]:
EVAL_ROOT = "/content/drive/MyDrive/PrivacyLens-Eval/실사-1차"   # 평가 사진 폴더(사진 + 정답 json)
YOLO_ROOT = "/content/drive/MyDrive/PrivacyLens-YOLO"            # YOLO 비교를 할 때만 씁니다
BRANCH = "main"     # 규칙을 고친 브랜치를 재 보려면 그 브랜치 이름
NOTE = ""           # 기록에 남길 메모. 예: "이름 규칙 보강 전"

## 1. 드라이브 연결 · 코드 받기 · 설치 (세션마다 한 번)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
os.makedirs(EVAL_ROOT, exist_ok=True)
if not os.path.isdir("/content/privacy-lens"):
    subprocess.run(["git", "clone", "-b", BRANCH,
                    "https://github.com/2026-usw-project/privacy-lens", "/content/privacy-lens"], check=True)
%cd /content/privacy-lens
!git fetch -q origin && git checkout -q $BRANCH && git pull -q
!pip -q install -r requirements.txt
# PaddleOCR: 3.3.x 는 CPU 추론 오류가 있어 3.2.2 로 고정(README 참고)
!pip -q install paddlepaddle==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
!pip -q install paddleocr
!python run.py --check

## 2. 처음 준비 — 정답 입력 도구와 촬영용 소품

- 사진 폴더에 `정답입력도구.html` 이 생깁니다. **각자 컴퓨터에서** 드라이브 폴더의 이 파일을 크롬으로 열고 '폴더 열기'로 같은 폴더를 고르세요.
- `소품_인쇄용.pdf` 를 A4 실제 크기로 인쇄해 잘라 찍으면, 정답 입력 도구에서 소품 번호만 골라 정답을 채울 수 있습니다(`props.json`).
- 이미 있는 정답 파일은 덮어쓰지 않습니다.

In [ ]:
!python -m samples.props "$EVAL_ROOT"
!python -m eval.failures init "$EVAL_ROOT"

## 3. 평가 — 지금 서비스(OCR + 규칙)

놓친 항목을 ① 못 찾음 ② 잘못 읽음 ③ 판단 실패로 나눕니다. 결과 숫자는 `평가기록.csv` 에 한 줄씩 쌓입니다.
글자 내용은 찍지 않습니다(`--show-text` 를 붙이면 보이지만 그 출력은 공유하지 마세요).

In [ ]:
!PL_YOLO=off python -m eval.failures run "$EVAL_ROOT" --note "OCR만 $NOTE"

## 4. (선택) YOLO 를 켰을 때와 비교

YOLO 공유 폴더의 current 모델로 같은 사진을 다시 잽니다. 3번과 4번 줄을 비교하면 YOLO 가 실제로 도움이 되는지 숫자로 나옵니다.

In [ ]:
!pip -q install ultralytics
ver = open(f"{YOLO_ROOT}/models/current.txt").read().strip()
weights = f"{YOLO_ROOT}/models/{ver}/best.pt"
print("YOLO", ver, weights)
!PL_YOLO_WEIGHTS="$weights" python -m eval.failures run "$EVAL_ROOT" --note "YOLO {ver} $NOTE"

## 5. 기록 보기

In [ ]:
import pandas as pd
p = f"{EVAL_ROOT}/평가기록.csv"
pd.read_csv(p, encoding="utf-8-sig") if os.path.exists(p) else "아직 기록이 없습니다."